# Tuần 2 — PhoBERT Multi-task Finetune (Anti-OOM)
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Notebook này dựa trên notebook PhoBERT training ban đầu và chỉ sửa cấu hình để hạn chế OOM CUDA:
- `max_seq_len = 256`
- `batch_size = 4`
- `grad_accumulation_steps = 8`

> Chạy theo thứ tự từ Cell 1 đến Cell 9.

In [ ]:
# ============================================================
# Cell 1
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    if vram < 14:
        print("VRAM < 14GB")
else:
    raise RuntimeError("Không có GPU!")

torch.cuda.empty_cache()
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# Install packages
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print("Dependencies installed")

GPU: Tesla T4
   VRAM: 15.6 GB
PyTorch: 2.10.0+cu128 | CUDA: 12.8
Dependencies installed


In [ ]:
# ============================================================
# Cell 2
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "feature/change-phobert"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("Clone completed")
else:
    print(f"Repo đã tồn tại tại {PROJECT_DIR} — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

# Chuyển vào thư mục project
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
sys.path.insert(0, "code/data_processing")
sys.path.insert(0, "code/phobert")

# Kiểm tra cấu trúc repo
print("\nCấu trúc repo:")
!ls -la
!ls code/data_processing/ code/phobert/

Repo đã tồn tại tại /kaggle/working/absa-project — pulling latest...
From https://github.com/vudinhminh08/NLP-project-master-study
 * branch            feature/change-phobert -> FETCH_HEAD
Already up to date.
Working dir: /kaggle/working/absa-project

Cấu trúc repo:
total 660
drwxr-xr-x 9 root root   4096 Apr 22 09:37  .
drwxr-xr-x 3 root root   4096 Apr 22 09:37  ..
-rw-r--r-- 1 root root    162 Apr 22 09:37 '~$oup16_Report_v6.docx'
-rw-r--r-- 1 root root    162 Apr 22 09:37 '~$oup16_Report_v8.docx'
drwxr-xr-x 2 root root   4096 Apr 22 09:37  app
drwxr-xr-x 7 root root   4096 Apr 22 09:37  code
drwxr-xr-x 2 root root   4096 Apr 22 09:37  data
drwxr-xr-x 2 root root   4096 Apr 22 09:37  docs
drwxr-xr-x 8 root root   4096 Apr 22 10:07  .git
-rw-r--r-- 1 root root   1372 Apr 22 09:37  .gitignore
-rw-r--r-- 1 root root 612522 Apr 22 09:37  Group16_Report_v8.docx
drwxr-xr-x 2 root root   4096 Apr 22 09:37  notebooks
drwxr-xr-x 6 root root   4096 Apr 22 09:37  outputs
-rw-r--r-- 1 root root

In [ ]:
# ============================================================
# Cell 3
# ============================================================
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("Data downloaded")
else:
    print("Data available")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows × {df.shape[1]} cols")

Data available
  train: 3000 rows × 35 cols
  dev: 2000 rows × 35 cols
  test: 600 rows × 35 cols


In [ ]:
# ============================================================
# Cell 4
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv"):
    print("Cache available (data/*_preprocessed.csv)")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Original : {s['Review'][:80]}")
    print(f"  Processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("Chưa có cache — chạy preprocessing với VnCoreNLP...")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  {split}: {len(df)} rows processed")
    segmenter.close()
    print("Preprocessing completed")

Cache available (data/*_preprocessed.csv)
  Original : Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.
  Processed: Rộng_rãi khách_sạn mới nhưng rất vắng . Các dịch_vụ chất_lượng chưa cao và thiếu


In [ ]:
# ============================================================
# Cell 5
# ============================================================
import json, os

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = '' if float(w) > 10 else ''
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

print('\n=== Train Config ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')

# Verify config values
# assert TRAIN_CONFIG['learning_rate'] == 1e-4, f"LR sai: {TRAIN_CONFIG['learning_rate']}"
# assert TRAIN_CONFIG['optimizer'] == 'Adam', f"Optimizer sai: {TRAIN_CONFIG['optimizer']}"
# assert PHOBERT_MODEL_NAME == 'vinai/phobert-base-v2', f"Model sai: {PHOBERT_MODEL_NAME}"
print(f'\n  ZERO_TRAIN_ASPECTS: {ZERO_TRAIN_ASPECTS}')
print('\nConfig loaded')


=== Encoder Config ===
  recommended_max_seq_len: 384
  p99_word_count: 243
  encoder_option: concat_4_layers
  encoder_hidden_size: 3072

=== Global Class Weights ===
  absent      : 1.0x
  positive    : 8.6x
  negative    : 28.0x
  neutral     : 154.5x

=== Train Config ===
  learning_rate: 0.0001
  warmup_ratio: 0.15
  batch_size: 8
  grad_accumulation_steps: 2
  max_epochs: 20
  early_stop_patience: 7
  dropout: 0.2
  focal_gamma: 2.0
  optimizer: AdamW
  scheduler: cosine_warmup
  seed: 42
  max_seq_len: 256
  head_lr_mult: 5.0
  layerwise_lr_decay: 0.92
  weight_decay: 0.01
  rare_aspect_mult: 2.0
  use_rare_oversampling: True
  rare_oversample_alpha: 2.0
  rare_oversample_power: 1.0
  rare_presence_pos_mult: 1.4
  rare_sentiment_mult: 1.25
  attn_dim: 128
  use_gradient_checkpointing: True
  mc_dropout_passes: 3
  use_attention_pooling: True
  use_split_loss: True
  lambda_presence: 1.0
  lambda_sentiment: 1.0
  presence_threshold: 0.5
  tune_presence_threshold: True
  presence_

In [ ]:
# ============================================================
# Cell 6
# ============================================================
import torch
torch.cuda.empty_cache()
print(f"VRAM free before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics = main(encoder_option='concat_4_layers', use_amp=True)

print('\n' + '='*55)
print('MAIN RUN RESULTS')
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print('='*55)


VRAM free before training: 15.6 GB
[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    concat_4_layers
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[Tokenizer] Loaded 

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 375 batches, rare_oversampling=ON (alpha=2.0, power=1.0)
[DataLoader] dev: 2000 samples, 250 batches
[DataLoader] test: 600 samples, 75 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (concat_4_layers)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[Model] 457,095,978 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=564
[LR] AdamW encoder_lr=1.00e-04, head_lr=5.00e-04, wd=1.00e-02, llrd=0.920, scheduler=cosine_warmup
[AMP] Mixed precision: ON
[EMA] OFF (decay=0.9990)
[Config] encoder=concat_4_layers, seq_len=256, batch=8×2=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 18
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
HOTEL#MISCELLANEOUS                  0.3027    0.7000    0.1931    0.1714    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
* FACILITIES#MISCELLANEOUS           0.4444    1.0000    0.2857    0.4286    7
FACILITIES#GENERAL                   0.5672    0.5278    0.6129    0.2482    62
FOOD&DRINKS#PRICES                   0.6000    0.7143    0.5172    0.4055    29
ROOM_AMENITIES


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
* FACILITIES#MISCELLANEOUS           0.2500    1.0000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3118    0.7073    0.2000    0.1757    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
FACILITIES#GENERAL                   0.5649    0.5362    0.5968    0.2482    62
FOOD&DRINKS#PRICES                   0.5882    0.6818    0.5172    0.4055    29
ROOM_AMENITIES


  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
* FACILITIES#MISCELLANEOUS           0.2500    1.0000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3118    0.7073    0.2000    0.1757    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
ROOM_AMENITIES#QUALITY               0.5839    0.4943    0.7131    0.5438    122
FACILITIES#GENERAL                   0.5865    0.5493    0.6290    0.2526    62
FOOD&DRINKS#P


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.3298    0.7209    0.2138    0.1839    145
* FOOD&DRINKS#MISCELLANEOUS          0.3333    0.4286    0.2727    0.2000    11
* FACILITIES#MISCELLANEOUS           0.4444    1.0000    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4821    0.6279    0.3913    0.3617    69
ROOMS#QUALITY                        0.5000    0.3571    0.8333    0.6000    6
FOOD&DRINKS#PRICES                   0.5660    0.6250    0.5172    0.4055    29
FACILITIES#GENERAL                   0.5694    0.5000    0.6613    0.2612    62
ROOM_AMENITIES#QU


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.2410    0.6667    0.1471    0.1212    68
FACILITIES#COMFORT                   0.4390    0.6000    0.3462    0.1905    26
FACILITIES#CLEANLINESS               0.4444    0.5000    0.4000    0.2667    5
* FOOD&DRINKS#MISCELLANEOUS          0.5000    1.0000    0.3333    0.3333    3
HOTEL#QUALITY                        0.5517    0.5000    0.6154    0.4667    13
FACILITIES#GENERAL                   0.5965    0.4722    0.8095    0.9608    21
ROOMS#QUALITY     

In [ ]:
# ============================================================
# Cell 7
# ============================================================
# Chứng minh kỹ thuật concat 4 layers có đóng góp thực sự
# (so sánh 3072 dim vs 768 dim trong báo cáo)
# Chạy SAU khi Cell 6 đã hoàn tất

import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

print("\n" + "="*55)
print("ABLATION SUMMARY")
print(f"  concat_4_layers: {test_metrics['macro_combined_f1']:.4f}  ← main architecture")
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"  Gain từ concat:  {gain*100:+.2f}%  {'concat tốt hơn trong lần chạy này' if gain > 0 else 'cls_only tốt hơn trong lần chạy này'}")
print("="*55)

[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    cls_only
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Tokenizer] Loaded 

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 375 batches, rare_oversampling=ON (alpha=2.0, power=1.0)
[DataLoader] dev: 2000 samples, 250 batches
[DataLoader] test: 600 samples, 75 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (cls_only)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[Model] 155,364,138 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=564
[LR] AdamW encoder_lr=1.00e-04, head_lr=5.00e-04, wd=1.00e-02, llrd=0.920, scheduler=cosine_warmup
[AMP] Mixed precision: ON
[EMA] OFF (decay=0.9990)
[Config] encoder=cls_only, seq_len=256, batch=8×2=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 18
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3299    0.6531    0.2207    0.1875    145
ROOMS#QUALITY                        0.3333    0.2222    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.3636    0.5000    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4567    0.5000    0.4203    0.3807    69
FOOD&DRINKS#PRICES                   0.5283    0.5833    0.4828    0.3848    29
FACILITIES#GENERAL                   0.5532    0.4937    0.6290    0.2680    62
ROOM_AMENITIES


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3141    0.6522    0.2069    0.1791    145
ROOMS#QUALITY                        0.3636    0.2500    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.4000    0.6667    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4538    0.5400    0.3913    0.3617    69
FACILITIES#GENERAL                   0.5401    0.4933    0.5968    0.2596    62
FOOD&DRINKS#PRICES                   0.5490    0.6364    0.4828    0.3848    29
ROOM_AMENITIES


  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3141    0.6522    0.2069    0.1791    145
ROOMS#QUALITY                        0.3636    0.2500    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.4000    0.6667    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4576    0.5510    0.3913    0.3617    69
FOOD&DRINKS#PRICES                   0.5490    0.6364    0.4828    0.3848    29
FACILITIES#GENERAL                   0.5507    0.5000    0.6129    0.2639    62
ROOM_AMENITIES


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3886    0.6212    0.2828    0.2360    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.5000    0.4800    0.5217    0.4285    69
* FACILITIES#MISCELLANEOUS           0.5455    0.7500    0.4286    0.5000    7
FOOD&DRINKS#PRICES                   0.5517    0.5517    0.5517    0.4279    29
ROOM_AMENITIES#QUALITY               0.5569    0.4387    0.7623    0.5656    122
FACILITIES#GENER


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.3226    0.6000    0.2206    0.1769    68
* FOOD&DRINKS#MISCELLANEOUS          0.5000    1.0000    0.3333    0.3333    3
HOTEL#QUALITY                        0.5000    0.5455    0.4615    0.3704    13
FACILITIES#CLEANLINESS               0.5455    0.5000    0.6000    0.3333    5
FACILITIES#GENERAL                   0.5926    0.4848    0.7619    0.9495    21
ROOM_AMENITIES#QUALITY               0.6043    0.5250    0.7119    0.4927    59
FACILITIES#PRICES 

In [ ]:
# ============================================================
# Cell 8
# ============================================================
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

# --- Loss ---
ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# --- F1 ---
ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Phân tích  ---
print(f"\nPhân tích learning curve ():")
print(f"  Best epoch:           {best_ep} / {len(history['train_loss'])}")
print(f"  Best Combined F1:     {history['best_combined_f1']:.4f}")

if len(history["train_loss"]) > best_ep:
    tloss_at   = history["train_loss"][best_ep - 1]
    tloss_after = history["train_loss"][best_ep]
    dloss_at   = history["dev_loss"][best_ep - 1]
    dloss_after = history["dev_loss"][best_ep]
    print(f"  Train loss epoch {best_ep}: {tloss_at:.4f} → epoch {best_ep+1}: {tloss_after:.4f} (tiếp tục giảm)")
    print(f"  Dev loss epoch {best_ep}:   {dloss_at:.4f} → epoch {best_ep+1}: {dloss_after:.4f} (tăng = overfit)")
    print(f"  → Early stopping đúng: dev F1 không cải thiện {history['config']['early_stop_patience']} epoch liên tiếp")

print(f"\n  Saved: outputs/eda/learning_curve.png")


Phân tích learning curve ():
  Best epoch:           17 / 20
  Best Combined F1:     0.6035
  Train loss epoch 17: 0.0054 → epoch 18: 0.0048 (tiếp tục giảm)
  Dev loss epoch 17:   0.3630 → epoch 18: 0.3723 (tăng = overfit)
  → Early stopping đúng: dev F1 không cải thiện 7 epoch liên tiếp

  Saved: outputs/eda/learning_curve.png


In [ ]:
# ============================================================
# Cell 9
# ============================================================
import os, json, shutil

# In summary report
report = "outputs/results/phobert_summary.md"
if os.path.exists(report):
    print(open(report, encoding="utf-8").read())
else:
    print("Chưa có report — đảm bảo Cell 6 đã chạy xong")

# Liệt kê tất cả files kết quả
print("\n=== Kết quả đã tạo ===")
result_files = []
for root, dirs, files in os.walk("outputs"):
    for f in files:
        if not f.endswith(".DS_Store"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            result_files.append(path)
            print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
print("\nTạo zip...")
shutil.make_archive("/kaggle/working/phobert_results", "zip", "outputs")
print("Zip: /kaggle/working/phobert_results.zip")
print("   → Kaggle: Output panel → Download")

# Tổng hợp kết quả chính
print("\n=== Tổng hợp kết quả ===")
if os.path.exists("outputs/results/phobert_test_metrics.json"):
    m = json.load(open("outputs/results/phobert_test_metrics.json"))
    print(f"PHOBERT_ACD_F1      = {m['macro_acd_f1']:.4f}")
    print(f"PHOBERT_SPC_F1      = {m['macro_spc_f1']:.4f}")
    print(f"PHOBERT_COMBINED_F1 = {m['macro_combined_f1']:.4f}")
if os.path.exists("outputs/results_cls_only/phobert_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/phobert_test_metrics.json"))
    print(f"PHOBERT_ABLATION_COMBINED_F1 = {m2['macro_combined_f1']:.4f}")

# Kết quả PhoBERT Multi-task

## Config thực tế
| Tham số | Giá trị |
|---------|---------|
| Encoder | concat_4_layers |
| MAX_SEQ_LEN | 256 |
| Batch size | 8 × 2 = 16 (effective) |
| Learning rate | 0.0001 |
| Warmup | 15% steps |
| Weight clip | 10.0 |
| Best epoch | 17 / 20 |

## Kết quả

| Split | ACD F1 | SPC F1 | Combined F1 |
|-------|--------|--------|-------------|
| Dev   | 0.6940 | 0.5311 | 0.6126 |
| **Test**  | **0.6827** | **0.5374** | **0.6101** |
| SOTA (Huynh 2022) | 0.8255 | — | 0.7732 |

## Phân tích Gap so với SOTA

- **ACD F1 gap:** 0.1428 (14.3%)
- **Combined F1 gap:** 0.1631 (16.3%)

### Nguyên nhân gap (phân tích):
1. **underthesea vs VnCoreNLP:** Dùng underthesea làm fallback → ~1-2% F1 loss
2. **ROOM_AMENITIES#PRICES:** 0 training samples → ACD F1 = 0 cho aspect này
3. **Neutral cực hiếm (weight=154→clip=10):** SPC F1 cho neutral thấp
4. **Dataset nhỏ (3000 train):** khó học tốt các aspect hiếm
5. **Single model:** báo cáo chính giữ bản single để dễ giải thí